In [0]:
from pyspark.sql.functions import trim, col, StringType
import pyspark.sql.functions as F

df = spark.table("dev_project.bronze.erp_loc_a101")

for filed in df.schema.fields:
    if isinstance(df.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


df = df.withColumn(
    "CNTRY", F.when(F.col("CNTRY").isNull(), "n/a").otherwise(F.col("CNTRY"))
)


df = df.withColumn(
    "CNTRY",
    F.when(col("CNTRY") == "DE", "Germany")
     .when(col("CNTRY").isin("US", "USA"), "United States")
     .when((col("CNTRY") == "") | col("cntry").isNull(), "n/a")
     .otherwise(col("CNTRY"))
)

name_map = {"CID": "customer_id", "CNTRY": "country"}

for old_name, new_name in name_map.items():
    df = df.withColumnRenamed(old_name, new_name)
df.display()

df.write.mode("overwrite").saveAsTable("dev_project.silver.erp_loc_a101")

In [0]:
df.select(F.col("country")).distinct().display()